In [1]:
import os
os.listdir("../data/raw")

['dvf_2021_33.csv.gz',
 'dvf_2022_33.csv.gz',
 'dvf_2023_33.csv.gz',
 'dvf_2024_33.csv.gz',
 'dvf_2025_33.csv.gz']

### Step A - Cleaning function
Applies the 8 steps from notebook 01 to one year and returns the clean table plus its funnel (row count after each step).

In [2]:
import os
import pandas as pd

YEARS = [2021, 2022, 2023, 2024, 2025]
KEEP = ["id_mutation", "date_mutation", "valeur_fonciere", "code_postal", "code_commune",
        "nom_commune", "type_local", "surface_reelle_bati", "nombre_pieces_principales",
        "surface_terrain", "longitude", "latitude"]

def clean_year(year):
    df = pd.read_csv(f"../data/raw/dvf_{year}_33.csv.gz", low_memory=False)
    f = {"1. Raw rows": len(df)}

    d = df[df["nature_mutation"] == "Vente"].copy()
    f["2. Nature = Vente"] = len(d)

    mixed = d.loc[d["type_local"] == "Local industriel. commercial ou assimilé", "id_mutation"].unique()
    d = d[~d["id_mutation"].isin(mixed)]
    f["3. No commercial premises"] = len(d)

    res = d[d["type_local"].isin(["Maison", "Appartement"])].copy()
    f["4. Houses/apartments rows"] = len(res)

    res = res.drop_duplicates(subset=["id_mutation", "type_local", "surface_reelle_bati", "nombre_pieces_principales"])
    f["5. After dropping duplicates"] = len(res)

    n = res.groupby("id_mutation")["id_mutation"].transform("size")
    res = res[n == 1].copy()
    f["6. One property per sale"] = len(res)

    res = res.dropna(subset=["valeur_fonciere", "surface_reelle_bati"]).copy()
    res["price_per_m2"] = res["valeur_fonciere"] / res["surface_reelle_bati"]
    f["7. Price and surface present"] = len(res)

    mask = (
        (res["valeur_fonciere"] >= 10_000)
        & res["surface_reelle_bati"].between(9, 500)
        & res["price_per_m2"].between(500, 15_000)
    )
    res = res[mask].copy()
    f["8. After outlier removal"] = len(res)

    res = res[KEEP + ["price_per_m2"]].copy()
    res["year"] = year
    return res, f

### Step B - Run on all years
Run the function on 2021-2025, stack the results into one table, and compare the funnels.
**Why compare:** a year whose funnel looks very different (e.g. many fewer rows) may signal incomplete or differently structured data.

In [3]:
frames, funnels = [], {}
for y in YEARS:
    clean_y, f = clean_year(y)
    frames.append(clean_y)
    funnels[y] = f

all_clean = pd.concat(frames, ignore_index=True)
pd.DataFrame(funnels)

,2021,2022,2023,2024,2025
1. Raw rows,118188,119809,92716,82927,111712
2. Nature = Vente,105934,108416,86087,76020,106253
3. No commercial premises,98644,99663,78969,69872,80476
4. Houses/apartments rows,36351,34538,26290,22897,27850
5. After dropping duplicates,31881,30578,23244,20126,24651
6. One property per sale,28513,27469,20905,18238,22349
7. Price and surface present,28497,27455,20890,18228,22339
8. After outlier removal,28097,27047,20545,17935,22032


### Step B - Results
Cleaned sales per year: 2021: 28,097 | 2022: 27,047 | 2023: 20,545 | 2024: 17,935 | 2025: 22,032 (total 115,656).
**Observation:** the funnels have the same shape in every year, except step 3 in 2025, which removes far more rows (about 25,800, versus 6,000-9,000 in other years). Investigated in Step E.

### Step C - Sanity checks
- Sales per year and median price/m² per year (does the trend look plausible?).
- Date range of each year (is 2025 a full year?).

In [4]:
all_clean["date_mutation"] = pd.to_datetime(all_clean["date_mutation"])
print(all_clean.groupby("year")["date_mutation"].agg(["min", "max"]))
all_clean.groupby("year")["price_per_m2"].agg(["count", "median"])

            min        max
year                      
2021 2021-01-02 2021-12-31
2022 2022-01-03 2022-12-31
2023 2023-01-02 2023-12-29
2024 2024-01-02 2024-12-31
2025 2025-01-02 2025-12-31


,count,median
year,,
2021,28097,3513.513514
2022,27047,3692.424737
2023,20545,3632.786885
2024,17935,3421.052632
2025,22032,3381.405112


### Step C - Results
All five years cover January to December, so 2025 is a full year.
Median price/m² (all Gironde): 3,514 (2021), 3,692 (2022), 3,633 (2023), 3,421 (2024), 3,381 (2025).
**Observation:** peak in 2022, about -8% by 2025. Sales volume fell sharply in 2023-2024 and partly recovered in 2025.
**Caveat:** this is a Gironde-wide median. It mixes houses, apartments and communes, so the trend can reflect a change in what was sold, not only a change in prices. The Bordeaux analysis will separate them.

### Step D - Save the combined clean table
Saved to `data/processed/dvf_33_2021_2025_clean.csv` (local only, `data/` is in `.gitignore`). It will be loaded into PostgreSQL next.

In [5]:
os.makedirs("../data/processed", exist_ok=True)
all_clean.to_csv("../data/processed/dvf_33_2021_2025_clean.csv", index=False)
len(all_clean)

115656

### Step E - Why does 2025 lose so many rows at step 3?
For each year I compare: number of sales, number of sales that include commercial premises, and the average number of rows per sale.
**Goal:** decide whether it is a real change (more mixed sales) or a data-structure change in the 2025 file.

In [6]:
rows = []
for y in YEARS:
    raw = pd.read_csv(f"../data/raw/dvf_{y}_33.csv.gz", low_memory=False,
                      usecols=["id_mutation", "nature_mutation", "type_local"])
    v = raw[raw["nature_mutation"] == "Vente"]
    sales = v["id_mutation"].nunique()
    mixed = v.loc[v["type_local"] == "Local industriel. commercial ou assimilé", "id_mutation"].nunique()
    rows.append({"year": y, "sales": sales, "sales_with_commercial": mixed,
                 "share_commercial": round(mixed / sales, 3),
                 "rows_per_sale": round(len(v) / sales, 2)})
pd.DataFrame(rows).set_index("year")

,sales,sales_with_commercial,share_commercial,rows_per_sale
year,,,,
2021,41450,1930,0.047,2.56
2022,40083,1942,0.048,2.70
2023,31220,1621,0.052,2.76
2024,27270,1470,0.054,2.79
2025,31695,1725,0.054,3.35


In [7]:
for y in [2024, 2025]:
    raw = pd.read_csv(f"../data/raw/dvf_{y}_33.csv.gz", low_memory=False, usecols=["nature_mutation", "type_local"])
    print(y)
    print(raw[raw["nature_mutation"] == "Vente"]["type_local"].value_counts(dropna=False), "\n")

2024
type_local
NaN                                         29914
Dépendance                                  19701
Maison                                      14275
Appartement                                  9631
Local industriel. commercial ou assimilé     2499
Name: count, dtype: int64 

2025
type_local
NaN                                         31744
Dépendance                                  29947
Maison                                      23771
Appartement                                 17210
Local industriel. commercial ou assimilé     3581
Name: count, dtype: int64 

